<p align="center">
  <a href="https://colab.research.google.com/github/samuel-c-santos/opensr-sentinel-pipeline/blob/main/OpenSR_Exploracao_Inferencia_Colab.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Google Colab" width="180">
  </a>
</p>

# ESA OpenSR - Super-Resolução de Sentinel-2 (LDSR-S2)
## Exploração e Inferência Interativa (10m para 2.5m / 4x)

Este notebook demonstra a aplicação prática do modelo **LDSR-S2 (Latent Diffusion Super-Resolution)** desenvolvido pela **ESA (Agência Espacial Europeia)** e pela comunidade **ESAOpenSR**.

### Características do Modelo:
- **100% Gratuito e Aberto**: Pesos públicos no Hugging Face sob licença Apache 2.0 (sem tokens ou limites proprietários).
- **Bandas VNIR Suportadas**: B4 (Vermelho), B3 (Verde), B2 (Azul) e B8 (Infravermelho Próximo).
- **Fator de Escala**: 4x (passa a resolução espacial de **10m/pixel para 2.5m/pixel**).
- **Pipeline Integrado**: Busca a melhor cena no Google Earth Engine (GEE), recorta o patch na coordenada de interesse e processa a super-resolução na GPU.


## 1. Instalação das Dependências
Instala o `opensr-model`, `rasterio` e bibliotecas geoespaciais necessárias.

> **Dica:** Ative a GPU no Google Colab em: *Ambiente de Execução > Alterar tipo de ambiente de execução > GPU (T4)*.

In [ ]:
# Instalação dos pacotes para OpenSR e Earth Engine
!pip install -q opensr-model omegaconf rasterio geemap earthengine-api

## 2. Autenticação e Inicialização do Google Earth Engine (GEE)
Utiliza o mesmo projeto GEE do seu fluxo de trabalho (`sentinel-landsat-download`).

In [ ]:
import ee

# Projeto GEE padrão
GEE_PROJECT = "sentinel-landsat-download"

try:
    ee.Initialize(project=GEE_PROJECT)
    print(f"✅ Google Earth Engine inicializado no projeto: {GEE_PROJECT}")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print(f"✅ Google Earth Engine autenticado e inicializado: {GEE_PROJECT}")

## 3. Seleção da Imagem Sentinel-2 Mais Limpa
Configuramos as coordenadas do ponto de interesse (padrão: IDEFLOR-Bio, Belém/PA) e filtramos a cena mais limpa na coleção `COPERNICUS/S2_SR_HARMONIZED`.

In [ ]:
# 1. Coordenadas do Ponto Central (Longitude, Latitude)
lon, lat = -48.4443399, -1.4248187  # IDEFLOR-Bio - Belém/PA
point = ee.Geometry.Point([lon, lat])

# 2. Intervalo de Busca e Limite de Nuvens
START_DATE = "2025-10-01"
END_DATE = "2026-03-01"
MAX_CLOUD_PERCENTAGE = 30

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD_PERCENTAGE))
    .sort("CLOUDY_PIXEL_PERCENTAGE")
)

n_images = collection.size().getInfo()
if n_images == 0:
    print("Nenhuma imagem encontrada no intervalo recente. Buscando a mais limpa histórica...")
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    image = ee.Image(collection.first())
else:
    image = ee.Image(collection.first())
    print(f"✅ Encontradas {n_images} imagens limpas no período!")

image_date = image.date().format("YYYY-MM-dd").getInfo()
cloud_cover = image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
image_id = image.get("system:index").getInfo()

print(f"📍 Cena selecionada: {image_id}")
print(f"📅 Data: {image_date}")
print(f"☁️ Cobertura de nuvens: {cloud_cover:.2f}%")

## 4. Download do Patch Sentinel-2 (10m)
Recorta uma janela em torno das coordenadas selecionadas contendo as 4 bandas VNIR de 10m: **B4 (Red), B3 (Green), B2 (Blue) e B8 (NIR)**.

In [ ]:
import os
import urllib.request
import rasterio
import numpy as np

os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

# Raio do buffer: 640m (gera uma janela de 1280m x 1280m = 128x128 pixels a 10m)
BUFFER_METERS = 640
region_box = point.buffer(BUFFER_METERS).bounds()

# Seleciona as bandas B4, B3, B2, B8
s2_patch = image.select(["B4", "B3", "B2", "B8"])

# Gera URL direta de download do GeoTIFF via Earth Engine
download_url = s2_patch.getDownloadURL({
    "scale": 10,
    "crs": "EPSG:3857",
    "region": region_box,
    "format": "GEO_TIFF"
})

input_tif_path = "/content/data/s2_input_10m.tif"
print("📥 Baixando patch de Sentinel-2 do GEE...")
urllib.request.urlretrieve(download_url, input_tif_path)

with rasterio.open(input_tif_path) as src:
    raw_lr_img = src.read()  # Shape: (4, Altura, Largura)
    input_profile = src.profile.copy()
    # O modelo LDSR-S2 requer dimensoes compativeis com a UNet (128x128).
    # A reprojecao geografica no GEE pode gerar pequenas variacoes (ex: 129x128).
    raw_lr_img = raw_lr_img[:, :128, :128]
    input_profile.update({"height": raw_lr_img.shape[1], "width": raw_lr_img.shape[2]})
    print(f"✅ Patch baixado com sucesso!")
    print(f"📐 Dimensões LR (10m): {raw_lr_img.shape[0]} bandas, {raw_lr_img.shape[1]}x{raw_lr_img.shape[2]} pixels")

## 5. Carregamento do Modelo ESA OpenSR (LDSR-S2)
Baixa a configuração oficial e os pesos pré-treinados do Hugging Face da ESA.

In [ ]:
from io import StringIO
import requests
import torch
from omegaconf import OmegaConf
import opensr_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Dispositivo ativo: {device.upper()}")
if device == "cpu":
    print("⚠️ ATENÇÃO: GPU não detectada. Habilite a GPU T4 no menu 'Ambiente de execução' para maior velocidade.")

# 1. Busca a configuração oficial de 10m
config_url = "https://raw.githubusercontent.com/ESAOpenSR/opensr-model/refs/heads/main/opensr_model/configs/config_10m.yaml"
response = requests.get(config_url)
config = OmegaConf.load(StringIO(response.text))

# 2. Instancia o modelo e carrega pesos pré-treinados da ESA
print("⏳ Carregando pesos pré-treinados do LDSR-S2 (HuggingFace)...")
model = opensr_model.SRLatentDiffusion(config, device=device)
model.load_pretrained(config.ckpt_version)
model.eval()
print("✅ Modelo ESA OpenSR pronto para inferência!")

## 6. Inferência de Super-Resolução (10m -> 2.5m / 4x)

In [ ]:
# 1. Normalização dos dados Sentinel-2 (Reflectância 0-10000 para 0-1)
raw_lr_img = raw_lr_img[:, :128, :128]  # Garante dimensão 128x128 compatível com a UNet
lr_tensor = torch.from_numpy(raw_lr_img).float().unsqueeze(0) / 10000.0
lr_tensor = lr_tensor.to(device)

# 2. Inferência
print(f"🔄 Executando super-resolução LDSR-S2 na entrada {tuple(lr_tensor.shape)}...")
with torch.no_grad():
    sr_tensor = model(lr_tensor)

# 3. Desnormalização de volta para escala uint16 (0-10000)
sr_numpy = sr_tensor.squeeze(0).cpu().numpy()
sr_numpy = np.clip(sr_numpy * 10000.0, 0, 10000).astype(np.uint16)

print("✨ Super-resolução concluída com sucesso!")
print(f"📐 Dimensões SR (2.5m): {sr_numpy.shape[0]} bandas, {sr_numpy.shape[1]}x{sr_numpy.shape[2]} pixels (Escala 4x)")

## 7. Salvar o GeoTIFF de Alta Resolução Georreferenciado (2.5m)
Ajusta a matriz afim (`Affine Transform`) para dividir o tamanho de pixel por 4 (10m / 4 = 2.5m) preservando o CRS EPSG:3857 original.

In [ ]:
from rasterio.transform import Affine

output_tif_path = "/content/output/s2_super_resolved_2_5m.tif"

# Escala 4x
scale_factor = 4
orig_transform = input_profile["transform"]
new_transform = Affine(
    orig_transform.a / scale_factor, orig_transform.b, orig_transform.c,
    orig_transform.d, orig_transform.e / scale_factor, orig_transform.f
)

out_profile = input_profile.copy()
out_profile.update({
    "height": sr_numpy.shape[1],
    "width": sr_numpy.shape[2],
    "transform": new_transform,
    "dtype": "uint16",
    "nodata": 0
})

with rasterio.open(output_tif_path, "w", **out_profile) as dst:
    dst.write(sr_numpy)

print(f"💾 GeoTIFF com Super-Resolução (2.5m) salvo em: {output_tif_path}")

## 8. Visualização Comparativa: Sentinel-2 (10m) vs ESA OpenSR (2.5m)
Compara lado a lado tanto em **Cor Verdadeira (RGB)** quanto em **Falsa-Cor (NIR-R-G)** para análise de vegetação e estruturas urbanas.

In [ ]:
import matplotlib.pyplot as plt

def to_rgb(img_bands, r_idx, g_idx, b_idx, p_min=2, p_max=98):
    rgb = np.stack([img_bands[r_idx], img_bands[g_idx], img_bands[b_idx]], axis=-1).astype(float)
    for c in range(3):
        band = rgb[..., c]
        v_min, v_max = np.percentile(band, (p_min, p_max))
        if v_max > v_min:
            rgb[..., c] = np.clip((band - v_min) / (v_max - v_min), 0, 1)
    return rgb

# Bandas no array: 0=B4(Red), 1=B3(Green), 2=B2(Blue), 3=B8(NIR)
rgb_lr = to_rgb(raw_lr_img, 0, 1, 2)
rgb_sr = to_rgb(sr_numpy, 0, 1, 2)

nir_lr = to_rgb(raw_lr_img, 3, 0, 1)  # Falsa-cor NIR-R-G
nir_sr = to_rgb(sr_numpy, 3, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

# 1. RGB Original 10m
axes[0, 0].imshow(rgb_lr)
axes[0, 0].set_title(f"Sentinel-2 Original (10m/px) - RGB\nData: {image_date}", fontsize=12, fontweight="bold")
axes[0, 0].axis("off")

# 2. RGB com Super-Resolução 2.5m
axes[0, 1].imshow(rgb_sr)
axes[0, 1].set_title("ESA OpenSR LDSR-S2 (2.5m/px) - RGB\nSuper-Resolução 4x", fontsize=12, fontweight="bold", color="darkgreen")
axes[0, 1].axis("off")

# 3. NIR Original 10m
axes[1, 0].imshow(nir_lr)
axes[1, 0].set_title("Sentinel-2 Original (10m/px) - Falsa Cor (NIR-R-G)", fontsize=12, fontweight="bold")
axes[1, 0].axis("off")

# 4. NIR com Super-Resolução 2.5m
axes[1, 1].imshow(nir_sr)
axes[1, 1].set_title("ESA OpenSR LDSR-S2 (2.5m/px) - Falsa Cor (NIR-R-G)", fontsize=12, fontweight="bold", color="darkgreen")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()